# Asset Failure - Survival Model - Pred
This notebook implements survival analysis models to predict asset failure times.
Survival analysis is particularly suitable for this task because it can handle:
1. Censored data (assets that haven't failed by the observation period end)
2. Time-to-event modeling (predicting when failure will occur)
3. Time-varying covariates (telemetry data that changes over time)

The workflow includes:
- Data aggregation from multiple sources (telemetry, tickets, assets, inventory)
- Feature engineering for survival analysis
- Multiple survival models (Kaplan-Meier, Cox PH, Random Survival Forest)
- Model evaluation and comparison

#### Author - Abhinav Paul

In [1]:
# Install required packages for survival analysis and data science
# lifelines: Comprehensive survival analysis library with Kaplan-Meier, Cox PH, etc.
# scikit-learn: Machine learning utilities for preprocessing and evaluation
# matplotlib/seaborn: Data visualization libraries
# scikit-survival: Advanced survival analysis methods including Random Survival Forest

! pip install lifelines scikit-learn matplotlib seaborn scikit-survival

In [2]:
# Import necessary libraries for data manipulation, visualization, and survival analysis

# Core data science libraries
import pandas as pd
# Configure pandas display options for better data exploration
pd.set_option("display.max_rows", 200)  # Show more rows in output
pd.set_option("display.max_columns", 50)  # Show more columns
pd.set_option("display.width", 1000)  # Wider display
pd.set_option("display.max_colwidth", None)  # Show full column content

import numpy as np  # Numerical operations
import os  # Operating system utilities
import pickle  # For serializing/deserializing Python objects

# Visualization libraries
import matplotlib.pyplot as plt  # Basic plotting
import seaborn as sns  # Statistical visualization

# Survival Analysis libraries
from lifelines.statistics import logrank_test  # Statistical tests for survival curves
from lifelines import KaplanMeierFitter, CoxPHFitter  # Non-parametric and semi-parametric models
from sksurv.ensemble import RandomSurvivalForest  # Machine learning approach to survival
from sksurv.util import Surv  # Utility for creating survival data structures
from sklearn.inspection import permutation_importance  # Feature importance evaluation
from sksurv.metrics import concordance_index_censored  # Model evaluation metric

# Data preprocessing
from sklearn.preprocessing import StandardScaler  # Feature scaling

# Date handling
from datetime import datetime, timedelta  # Date manipulation utilities

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings("ignore")

### Load Data

Load all necessary data sources for the survival analysis:
- inventory: Asset specifications and stock information
- telemetry: Time-series performance metrics for assets
- assets: Asset metadata including warranty and installation dates
- tickets: Support tickets indicating issues and maintenance activities

Each data source provides different dimensions needed for comprehensive survival modeling

In [3]:
# Load data from pickle files (efficient binary format for pandas DataFrames)
# Using pickle format preserves data types and is faster than CSV for large datasets
inventory = pd.read_pickle('/content/base_data/inventory.pkl')
telemetry = pd.read_pickle('/content/base_data/telemetry.pkl')
assets = pd.read_pickle('/content/base_data/assets.pkl')
tickets = pd.read_pickle('/content/base_data/tickets.pkl')

### Aggregated Analytical Dataset at Asset Client level

The goal is to create a unified dataset that combines information from all sources
at the appropriate granularity for survival analysis. We'll aggregate data to:
- Asset-client level (each asset for each client)
- Daily level (telemetry and ticket data aggregated by date)
- This allows us to track asset performance and failure events over time

Statistical reasoning: Survival analysis requires time-dependent covariates and
event data. Aggregating to daily level provides the temporal resolution needed
to model how asset conditions evolve toward failure.

#### Telemetry Aggregation

Telemetry data contains high-frequency performance metrics. We need to aggregate
these to daily level because:
1. Survival analysis works with discrete time intervals
2. Daily aggregation reduces noise while preserving meaningful patterns
3. Provides computational efficiency for large datasets

Statistical approach: Use mean for continuous metrics (CPU, memory, etc.) to
represent typical daily performance, and sum for count metrics (errors, failures)
to capture total daily burden on the asset.

In [4]:
# Define aggregation strategy for telemetry metrics
tel_agg_mx = {
    'cpu_utilization_percent' : 'mean',      # Average CPU usage represents typical load
    'memory_utilization_percent':'mean',     # Average memory usage indicates resource pressure
    'disk_utilization_percent':'mean',        # Average disk usage shows storage utilization
    'temperature_celsius':'mean',             # Average temperature indicates thermal stress
    'network_latency_ms':'mean',              # Average latency shows network performance
    'packet_loss_percent':'mean',             # Average packet loss indicates network quality
    'error_count':'sum',                      # Sum of errors captures total daily issues
    'failure_flag':'sum'                      # Sum of failures counts daily failure events
}

# Extract date from timestamp for daily aggregation
telemetry['telemetry_record_date'] = pd.to_datetime(telemetry['timestamp']).dt.date

# Group by asset, client, and date to create daily aggregated metrics
# This creates the time-dependent covariates needed for survival analysis
tel_agg = telemetry.groupby(['asset_id', 'client_id', 'telemetry_record_date']).agg(tel_agg_mx).reset_index(drop=False)

# Rename failure_flag to failure_count for clarity
tel_agg.rename(columns={'failure_flag':'failure_count'}, inplace=True)

In [5]:
# Examine aggregated telemetry data for a specific asset (AID_053)
# This validation step ensures aggregation worked correctly and shows:
# 1. Daily performance patterns for the asset
# 2. How metrics vary across different clients for the same asset
# 3. Presence of failure events (failure_count > 0)
tel_agg[tel_agg['asset_id'] == 'AID_053'].sort_values('telemetry_record_date').head()

,asset_id,client_id,telemetry_record_date,cpu_utilization_percent,memory_utilization_percent,disk_utilization_percent,temperature_celsius,network_latency_ms,packet_loss_percent,error_count,failure_count
31587,AID_053,C021,2023-01-01,81.222113,32.889289,90.954855,47.190867,107.243675,1.716625,19.0,0
31434,AID_053,C016,2023-01-01,66.407481,73.728408,67.406288,83.661472,96.515491,2.124910,12.0,0
31588,AID_053,C021,2023-01-05,91.243473,56.063039,92.228226,88.068999,23.554456,4.264910,12.0,1
31589,AID_053,C021,2023-01-06,77.136052,25.421653,53.199288,59.405809,68.311773,0.279547,1.0,0
31671,AID_053,C025,2023-01-09,73.655471,43.671441,29.304320,85.009862,67.307080,1.614539,15.0,0


In [6]:
# Verify the number of unique assets in telemetry data
# This confirms we have data for all assets in our inventory
# Expected: 64 unique assets based on the inventory data
tel_agg['asset_id'].nunique()

64

#### Tickets Aggregation

Ticket data represents maintenance events and user-reported issues.
For survival analysis, tickets are crucial because:
1. They indicate asset health problems that may precede failure
2. Different ticket types (P1, P2, etc.) represent varying severity levels
3. Response times and escalations indicate asset criticality
4. Ticket patterns can be leading indicators of impending failure

Statistical approach: Aggregate tickets by date range (create_date to close_date)
to capture the maintenance burden during specific periods.

In [7]:
# Create unique ticket identifier to check for duplicates
# Important for data quality - duplicate tickets would skew our analysis
tickets['file_year_ticket_id'] = tickets['file_year'].astype(str) + '_' + tickets['ticket_id'].astype(str)

# Data quality check: Ensure no duplicate tickets exist
# Duplicates would indicate data collection issues and bias the survival analysis
assert tickets['file_year_ticket_id'].count() - tickets['file_year_ticket_id'].nunique() == 0, f"""
Duplicate ticket_id detected!
Total rows: {tickets['file_year_ticket_id'].count()}
Unique IDs: {tickets['file_year_ticket_id'].nunique()}
Duplicates: {tickets['file_year_ticket_id'].count() - tickets['file_year_ticket_id'].nunique()}
"""

# Define aggregation strategy for ticket data
tickets_agg_mx = {
    'ticket_id' : 'count',                    # Total number of tickets
    'engineer_id' : 'count',                  # Number of engineers involved (indicates complexity)
    'escalation_flag':'sum',                  # Total escalations (severity indicator)
    'ticket_reopen_flag':'sum',               # Reopened tickets (recurrent issues)
    'first_response_time_minutes':'mean',     # Average response time (service quality)
    'resolution_time_minutes':'mean',          # Average resolution time (issue complexity)
    'sla_breach_flag':'sum',                  # SLA breaches (service failures)
    'ticket_priority_P1':'sum',               # Priority 1 tickets (critical issues)
    'ticket_priority_P2':'sum',               # Priority 2 tickets (high priority)
    'ticket_priority_P3':'sum',               # Priority 3 tickets (medium priority)
    'ticket_priority_P4':'sum',               # Priority 4 tickets (low priority)
    'issue_type_Access':'sum',                # Access-related issues
    'issue_type_Crash':'sum',                 # System crashes
    'issue_type_Failure':'sum',               # Failure reports
    'issue_type_Slow':'sum',                  # Performance issues
    'ticket_category_Hardware':'sum',          # Hardware-related tickets
    'ticket_category_Network':'sum',           # Network-related tickets
}

# Extract dates from timestamps for proper date-based aggregation
tickets['ticket_create_date'] = pd.to_datetime(tickets['ticket_created_timestamp']).dt.date
tickets['ticket_close_date'] = pd.to_datetime(tickets['ticket_close_timestamp']).dt.date

# One-hot encode categorical variables for aggregation
# This allows us to count different types of issues and priorities
for i in ['ticket_priority', 'issue_type', 'ticket_category']:
    encoded = pd.get_dummies(tickets[i], prefix=i)
    tickets = pd.concat([tickets.drop(columns=[i]), encoded], axis=1)

# Aggregate tickets by asset, client, and date range
# This creates time-dependent covariates representing maintenance burden
tickets_agg = tickets.groupby(['asset_id', 'client_id', 'ticket_create_date', 'ticket_close_date']).agg(tickets_agg_mx).reset_index(drop=False)

# Rename columns for clarity and consistency
tickets_agg.rename(columns={'ticket_id' : 'ticket_count',
                            'engineer_id' : 'engineer_worked_count',
                            'escalation_flag':'escalations_count',
                            'ticket_reopen_flag':'ticket_reopened_count',
                            'first_response_time_minutes':'avg_first_response_time_minutes',
                            'resolution_time_minutes':'avg_resolution_time_minutes',
                            'sla_breach_flag':'sla_breach_count',
                            'ticket_priority_P1':'P1_ticket_count',
                            'ticket_priority_P2':'P2_ticket_count',
                            'ticket_priority_P3':'P3_ticket_count',
                            'ticket_priority_P4':'P4_ticket_count',
                            'issue_type_Access':'Access_issue_count',
                            'issue_type_Crash':'Crash_issue_count',
                            'issue_type_Failure':'Failure_issue_count',
                            'issue_type_Slow':'Slow_issue_count',
                            'ticket_category_Hardware':'Hardware_category_count',
                            'ticket_category_Network':'Network_category_count',
                        }, inplace=True
                    )

In [8]:
# Examine aggregated ticket data for the same asset (AID_053)
# This shows the maintenance and issue patterns for the asset
# Key insights: ticket volumes, priorities, issue types, and resolution patterns
# These patterns are valuable predictors for survival analysis
tickets_agg[tickets_agg['asset_id'] == 'AID_053'].head(20).sort_values(['ticket_create_date','ticket_close_date']).head()

,asset_id,client_id,ticket_create_date,ticket_close_date,ticket_count,engineer_worked_count,escalations_count,ticket_reopened_count,avg_first_response_time_minutes,avg_resolution_time_minutes,sla_breach_count,P1_ticket_count,P2_ticket_count,P3_ticket_count,P4_ticket_count,Access_issue_count,Crash_issue_count,Failure_issue_count,Slow_issue_count,Hardware_category_count,Network_category_count
18397,AID_053,C007,2024-01-05,2024-01-06,1,1,1.0,0.0,41.0,1704.18,1.0,0,0,1,0,0,1,0,0,0,1
18398,AID_053,C007,2024-01-22,2024-01-22,1,1,0.0,0.0,45.0,204.07,0.0,1,0,0,0,0,1,0,0,1,0
18399,AID_053,C007,2024-01-23,2024-01-25,1,1,0.0,1.0,11.0,2029.20,0.0,0,0,0,1,1,0,0,0,1,0
18400,AID_053,C007,2024-01-27,2024-01-27,1,1,0.0,0.0,20.0,477.72,0.0,0,1,0,0,0,0,0,1,0,1
18401,AID_053,C007,2024-01-31,2024-02-01,1,1,1.0,0.0,26.0,1548.57,1.0,0,0,1,0,1,0,0,0,1,0


In [9]:
# Verify the number of unique assets in ticket data
# Should match telemetry data (64 assets) for complete coverage
tickets_agg['asset_id'].nunique()

64

#### Assets Warranty Aggregation

Asset metadata provides critical temporal boundaries for survival analysis:
1. Installation date: Start of observation period for each asset
2. Warranty expiry date: End of coverage period (potential censoring point)
3. Criticality level: Asset importance affecting maintenance priority

Statistical importance: These dates define the observation window for each asset,
which is essential for proper censoring handling in survival analysis.

In [10]:
# Map criticality levels to numerical values for prioritization
# This helps handle cases where assets have multiple records with different priorities
priority_map = {'Low': 1, 'Medium': 2, 'High': 3}

# Process assets data to get the most relevant record for each asset-client-year combination
# Statistical reasoning: When assets have multiple criticality levels, we want the highest
# priority (most critical) record for survival analysis, as critical assets may have
# different failure patterns and maintenance schedules
assets = (
    assets.assign(priority=assets['criticality_level'].map(priority_map))
          .sort_values(['asset_id', 'client_id', 'active_year', 'priority'], ascending=[True, True, True, False])
          .drop_duplicates(['asset_id', 'client_id', 'active_year'])
          .drop(columns='priority')
)

In [11]:
# Convert date columns to proper date format for temporal analysis
# These dates are critical for defining the observation periods in survival analysis and the failure event for out of warranty devices
assets['warranty_expiry_date'] = pd.to_datetime(assets['warranty_expiry_date']).dt.date
assets['installation_date'] = pd.to_datetime(assets['installation_date']).dt.date

# Create aggregated assets dataset with key temporal information
# Focus on dates that define the survival analysis observation window
assets_agg = assets[['asset_id', 'client_id', 'warranty_expiry_date', 'installation_date']].drop_duplicates().reset_index(drop=True)

In [12]:
# Examine assets data for a different asset (AID_016) to understand the data structure
# Shows how the same asset can have different warranty periods for different clients
# This is important because survival analysis needs to account for asset usage context
assets_agg[assets_agg['asset_id'] == 'AID_016'].head(20).sort_values(['asset_id','client_id'])

,asset_id,client_id,warranty_expiry_date,installation_date
100,AID_016,C004,2026-06-26,2023-06-27
101,AID_016,C005,2026-01-26,2023-01-27
102,AID_016,C008,2025-08-12,2022-08-13
103,AID_016,C013,2026-10-31,2023-11-01
104,AID_016,C015,2027-12-24,2024-12-24
105,AID_016,C016,2026-07-14,2023-07-15
106,AID_016,C022,2027-09-13,2024-09-13


In [13]:
# Verify the number of unique assets in assets data
# Should match other datasets (64 assets) for complete coverage
assets_agg['asset_id'].nunique()

64

#### Inventory Aggregation

Inventory data provides static asset characteristics that can influence survival:
1. Device type, manufacturer, model: Different hardware may have different reliability
2. Unit cost: More expensive assets may have better maintenance
3. Warehouse location: Environmental factors may affect asset lifespan
4. Lead time: Replacement time affects business impact of failures

In [14]:
# Examine the inventory data structure
# Shows static asset characteristics that will be used as baseline covariates
# in the survival analysis model
inventory.head()

,asset_id,device_type,manufacturer,model_number,warehouse_location,current_stock_quantity,reorder_threshold_quantity,unit_cost,safety_stock_quantity,lead_time_days,file_year
0,AID_001,Server,Dell,M001,WH1,56,40,2753.0,17,6,2023
1,AID_002,Server,Dell,M002,WH1,76,40,4967.0,12,12,2023
2,AID_003,Server,Dell,M003,WH2,58,40,744.0,12,6,2023
3,AID_004,Server,Dell,M004,WH1,179,40,717.0,27,6,2023
4,AID_005,Server,Cisco,M001,WH2,106,40,4179.0,28,7,2023


In [15]:
# Get the latest inventory record for each asset
# Statistical reasoning: Use the most recent inventory data as it represents the current
# asset specifications and stock levels, which are most relevant for current survival predictions

latest_inventory = inventory.sort_values(['asset_id', 'file_year']).groupby('asset_id').tail(1).reset_index(drop=True)

# Data quality check: Ensure we have exactly one record per asset
# Duplicate asset records would indicate data quality issues
assert latest_inventory['asset_id'].count() - latest_inventory['asset_id'].nunique() == 0, f"""
Duplicate asset_id detected!
Total rows: {latest_inventory['asset_id'].count()}
Unique IDs: {latest_inventory['asset_id'].nunique()}
Duplicates: {latest_inventory['asset_id'].count() - latest_inventory['asset_id'].nunique()}
"""

# Verify the latest inventory data for our test asset (AID_053)
latest_inventory[latest_inventory['asset_id'] == 'AID_053'].head(20).sort_values(['asset_id'])

,asset_id,device_type,manufacturer,model_number,warehouse_location,current_stock_quantity,reorder_threshold_quantity,unit_cost,safety_stock_quantity,lead_time_days,file_year
52,AID_053,Switch,Cisco,M001,WH2,119,40,4535.99,19,10,2025


In [16]:
# Verify that we have the latest inventory data for all assets
# All assets should have 2025 data, indicating we're using the most recent information
latest_inventory.file_year.value_counts()

,count
file_year,
2025,64


### Merge all DataFrames at Asset-Client ID level

Now we'll merge all the processed datasets to create a comprehensive Analytical Dataset (ADS)
for survival analysis. The merge strategy is designed to:
1. Preserve temporal relationships between telemetry and tickets
2. Maintain asset-specific characteristics as baseline covariates
3. Create time-dependent observations suitable for survival modeling

Statistical importance: Proper merging ensures we have the right covariates at the right
time points, which is crucial for accurate survival predictions.

#### Add prefixes to identify data sources

Prefix naming convention for clarity and to avoid column name conflicts:
- tel_: telemetry metrics (time-varying performance data)
- t_: ticket metrics (maintenance and issue data)
- a_: asset metrics (warranty and installation dates)
- i_: inventory metrics (static asset characteristics)

This naming strategy improves code readability and debugging in survival analysis

In [17]:
# Apply prefixes to all columns except key identifiers
# This creates clear, non-conflicting column names for the merged dataset
tel_prefixed = tel_agg.rename(columns=lambda x: f"tel_{x}" if x not in ['asset_id', 'client_id', 'telemetry_record_date'] else x)
tickets_prefixed = tickets_agg.rename(columns=lambda x: f"t_{x}" if x not in ['asset_id', 'client_id', 'ticket_create_date', 'ticket_close_date'] else x)
assets_prefixed = assets_agg.rename(columns=lambda x: f"a_{x}" if x not in ['asset_id', 'client_id'] else x)
latest_inventory = latest_inventory.rename(columns=lambda x: f"i_{x}" if x not in ['asset_id'] else x)

# Sort data for proper temporal ordering
# Critical for survival analysis to maintain chronological sequence
tel_prefixed = tel_prefixed.sort_values(['asset_id', 'client_id', 'telemetry_record_date'])
tickets_prefixed = tickets_prefixed.sort_values(['asset_id', 'client_id', 'ticket_create_date', 'ticket_close_date'])

# Display dataset shapes to understand data volume
print(tel_prefixed.shape, tickets_prefixed.shape, assets_prefixed.shape, latest_inventory.shape)

(37649, 11) (21773, 21) (417, 4) (64, 11)


#### Merge tables into one Analytical Dataset (ADS)

The merge strategy is carefully designed for survival analysis:
1. First merge telemetry with tickets based on temporal alignment
2. Then add asset temporal boundaries (warranty, installation)
3. Finally add static inventory characteristics

Statistical approach: This creates time-dependent observations where each row
represents an asset's state on a specific day, with corresponding covariates
and event information needed for survival modeling.

In [18]:
# Create next ticket date for temporal alignment
# This helps associate telemetry data with the relevant ticket period
# Statistical reasoning: Each telemetry reading should be linked to the ticket
# period it belongs to, creating proper time-dependent covariates
tickets_prefixed['next_ticket_create_date'] = (
    tickets_prefixed
    .groupby(['asset_id', 'client_id'])['ticket_create_date']
    .shift(-1)
).fillna(pd.to_datetime('today'))

# Merge telemetry with tickets on asset-client keys
# Left join ensures we keep all telemetry records
final_ads = tel_prefixed.merge(
    tickets_prefixed,
    on=['asset_id', 'client_id'],
    how='left'
)

# Convert dates to proper format for temporal filtering
final_ads['ticket_create_date'] = pd.to_datetime(final_ads['ticket_create_date']).dt.date
final_ads['next_ticket_create_date'] = pd.to_datetime(final_ads['next_ticket_create_date']).dt.date

# Filter telemetry to match ticket periods
# This ensures telemetry data is associated with the correct maintenance context
# Critical for survival analysis to have accurate time-dependent covariates
final_ads = final_ads[
    (final_ads['telemetry_record_date'] >= final_ads['ticket_create_date']) &
    (final_ads['telemetry_record_date'] < final_ads['next_ticket_create_date'])
]

# Display the merged dataset structure
final_ads.head()

,asset_id,client_id,telemetry_record_date,tel_cpu_utilization_percent,tel_memory_utilization_percent,tel_disk_utilization_percent,tel_temperature_celsius,tel_network_latency_ms,tel_packet_loss_percent,tel_error_count,tel_failure_count,ticket_create_date,ticket_close_date,t_ticket_count,t_engineer_worked_count,t_escalations_count,t_ticket_reopened_count,t_avg_first_response_time_minutes,t_avg_resolution_time_minutes,t_sla_breach_count,t_P1_ticket_count,t_P2_ticket_count,t_P3_ticket_count,t_P4_ticket_count,t_Access_issue_count,t_Crash_issue_count,t_Failure_issue_count,t_Slow_issue_count,t_Hardware_category_count,t_Network_category_count,next_ticket_create_date
60,AID_001,C005,2024-01-04,67.171200,21.245898,69.233731,48.310766,146.732377,2.363257,19.0,0,2024-01-04,2024-01-06,1,1,0.0,1.0,13.0,3333.57,0.0,0,0,0,1,0,0,0,1,0,1,2024-01-11
120,AID_001,C005,2024-01-05,35.336757,66.002886,49.620884,53.319463,62.649608,4.790338,6.0,0,2024-01-04,2024-01-06,1,1,0.0,1.0,13.0,3333.57,0.0,0,0,0,1,0,0,0,1,0,1,2024-01-11
180,AID_001,C005,2024-01-08,70.578901,47.685597,27.861405,89.365066,19.929828,2.557261,2.0,0,2024-01-04,2024-01-06,1,1,0.0,1.0,13.0,3333.57,0.0,0,0,0,1,0,0,0,1,0,1,2024-01-11
241,AID_001,C005,2024-01-12,73.900398,89.154157,87.782479,81.979576,147.011057,3.381585,0.0,0,2024-01-11,2024-01-14,1,1,1.0,1.0,8.0,4656.12,1.0,0,0,0,1,0,0,1,0,1,0,2024-01-21
301,AID_001,C005,2024-01-15,52.580138,61.416155,78.709126,87.160724,120.441571,0.037180,17.0,0,2024-01-11,2024-01-14,1,1,1.0,1.0,8.0,4656.12,1.0,0,0,0,1,0,0,1,0,1,0,2024-01-21


In [19]:
# Track dataset size changes through merging process
base = final_ads.shape

# Merge with asset temporal information (warranty and installation dates)
# These dates define the observation window for survival analysis
final_ads = final_ads.merge(
    assets_prefixed,
    on=['asset_id', 'client_id'],
    how='left'
)

# Handle missing warranty information using asset-level averages
# Statistical approach: Use mean warranty window as reasonable imputation
# This preserves the temporal structure while handling missing data
back_up_warranty = assets.groupby('asset_id').warranty_window.mean().reset_index()
back_up_warranty.rename(columns={'warranty_window' : 'warranty_window_asset_level_mean'}, inplace=True)

final_ads = final_ads.merge(
    back_up_warranty,
    on=['asset_id'],
    how='left'
)

# Impute missing warranty dates using asset-level averages
# Statistical reasoning: Assets with missing warranty data get estimated dates
# based on their asset type's typical warranty period
final_ads['a_warranty_expiry_date'] = np.where(
    final_ads['a_warranty_expiry_date'].isna(),
    pd.to_datetime('today') + pd.to_timedelta(final_ads['warranty_window_asset_level_mean'], unit='d'),
    final_ads['a_warranty_expiry_date']
)

final_ads['a_installation_date'] = np.where(
    final_ads['a_installation_date'].isna(),
    pd.to_datetime(final_ads['a_warranty_expiry_date']) - pd.to_timedelta(final_ads['warranty_window_asset_level_mean'], unit='d'),
    final_ads['a_installation_date']
)

# Clean up temporary columns
final_ads.drop(columns=['warranty_window_asset_level_mean'], inplace=True)

# Filter data to valid observation periods
# Critical for survival analysis: only include data within warranty/active periods
final_ads = final_ads[
    (pd.to_datetime(final_ads['telemetry_record_date']) >= pd.to_datetime(final_ads['a_installation_date'])) &
    (pd.to_datetime(final_ads['telemetry_record_date']) < pd.to_datetime(final_ads['a_warranty_expiry_date']))
]

# Report data loss/gain from merging
print(base, assets_agg.shape, final_ads.shape)
print(f"Change in final_ads post asset merge = {base[0] - final_ads.shape[0]}, ({((base[0] - final_ads.shape[0])*100.0/base[0]):.2f} %)")
del base

# Display the final merged dataset
final_ads.head()

(36947, 31) (417, 4) (40053, 33)
Change in final_ads post asset merge = -3106, (-8.41 %)


,asset_id,client_id,telemetry_record_date,tel_cpu_utilization_percent,tel_memory_utilization_percent,tel_disk_utilization_percent,tel_temperature_celsius,tel_network_latency_ms,tel_packet_loss_percent,tel_error_count,tel_failure_count,ticket_create_date,ticket_close_date,t_ticket_count,t_engineer_worked_count,t_escalations_count,t_ticket_reopened_count,t_avg_first_response_time_minutes,t_avg_resolution_time_minutes,t_sla_breach_count,t_P1_ticket_count,t_P2_ticket_count,t_P3_ticket_count,t_P4_ticket_count,t_Access_issue_count,t_Crash_issue_count,t_Failure_issue_count,t_Slow_issue_count,t_Hardware_category_count,t_Network_category_count,next_ticket_create_date,a_warranty_expiry_date,a_installation_date
0,AID_001,C005,2024-01-04,67.171200,21.245898,69.233731,48.310766,146.732377,2.363257,19.0,0,2024-01-04,2024-01-06,1,1,0.0,1.0,13.0,3333.57,0.0,0,0,0,1,0,0,0,1,0,1,2024-01-11,2026-04-27,2023-04-28
1,AID_001,C005,2024-01-05,35.336757,66.002886,49.620884,53.319463,62.649608,4.790338,6.0,0,2024-01-04,2024-01-06,1,1,0.0,1.0,13.0,3333.57,0.0,0,0,0,1,0,0,0,1,0,1,2024-01-11,2026-04-27,2023-04-28
2,AID_001,C005,2024-01-08,70.578901,47.685597,27.861405,89.365066,19.929828,2.557261,2.0,0,2024-01-04,2024-01-06,1,1,0.0,1.0,13.0,3333.57,0.0,0,0,0,1,0,0,0,1,0,1,2024-01-11,2026-04-27,2023-04-28
3,AID_001,C005,2024-01-12,73.900398,89.154157,87.782479,81.979576,147.011057,3.381585,0.0,0,2024-01-11,2024-01-14,1,1,1.0,1.0,8.0,4656.12,1.0,0,0,0,1,0,0,1,0,1,0,2024-01-21,2026-04-27,2023-04-28
4,AID_001,C005,2024-01-15,52.580138,61.416155,78.709126,87.160724,120.441571,0.037180,17.0,0,2024-01-11,2024-01-14,1,1,1.0,1.0,8.0,4656.12,1.0,0,0,0,1,0,0,1,0,1,0,2024-01-21,2026-04-27,2023-04-28


In [20]:
# Final data quality check: Ensure no missing values remain
# Critical for survival analysis models which typically don't handle missing data well
na = final_ads.isna().sum().reset_index()

assert na[na[0]!=0].shape[0] == 0, f"""
Missing values found!
Total rows: {final_ads.shape[0]}
Missing values: {na[na[0]!=0].shape[0]}
"""

In [21]:
# Validate the final dataset structure for our test asset (AID_053)
# This confirms that all merging worked correctly and we have a complete
# time-dependent dataset ready for survival analysis
final_ads[final_ads['asset_id'] == 'AID_053'].head(20).sort_values(['telemetry_record_date'])

,asset_id,client_id,telemetry_record_date,tel_cpu_utilization_percent,tel_memory_utilization_percent,tel_disk_utilization_percent,tel_temperature_celsius,tel_network_latency_ms,tel_packet_loss_percent,tel_error_count,tel_failure_count,ticket_create_date,ticket_close_date,t_ticket_count,t_engineer_worked_count,t_escalations_count,t_ticket_reopened_count,t_avg_first_response_time_minutes,t_avg_resolution_time_minutes,t_sla_breach_count,t_P1_ticket_count,t_P2_ticket_count,t_P3_ticket_count,t_P4_ticket_count,t_Access_issue_count,t_Crash_issue_count,t_Failure_issue_count,t_Slow_issue_count,t_Hardware_category_count,t_Network_category_count,next_ticket_create_date,a_warranty_expiry_date,a_installation_date
35368,AID_053,C007,2024-01-05,74.493250,50.510745,78.678625,61.449120,90.820059,1.839542,27.0,1,2024-01-05,2024-01-06,1,1,1.0,0.0,41.0,1704.18,1.0,0,0,1,0,0,1,0,0,0,1,2024-01-22,2026-12-19,2023-12-20
35369,AID_053,C007,2024-01-13,42.539903,20.117163,28.632376,63.084264,103.301998,0.258377,1.0,0,2024-01-05,2024-01-06,1,1,1.0,0.0,41.0,1704.18,1.0,0,0,1,0,0,1,0,0,0,1,2024-01-22,2026-12-19,2023-12-20
35370,AID_053,C007,2024-01-22,31.451768,72.897748,67.670786,39.409272,61.610585,3.039540,15.0,0,2024-01-22,2024-01-22,1,1,0.0,0.0,45.0,204.07,0.0,1,0,0,0,0,1,0,0,1,0,2024-01-23,2026-12-19,2023-12-20
35371,AID_053,C007,2024-01-24,67.023687,63.656638,94.205545,73.065436,138.489961,2.287923,5.0,0,2024-01-23,2024-01-25,1,1,0.0,1.0,11.0,2029.20,0.0,0,0,0,1,1,0,0,0,1,0,2024-01-27,2026-12-19,2023-12-20
35372,AID_053,C007,2024-01-25,67.292266,28.897697,47.956766,71.822064,100.570987,3.396641,18.0,0,2024-01-23,2024-01-25,1,1,0.0,1.0,11.0,2029.20,0.0,0,0,0,1,1,0,0,0,1,0,2024-01-27,2026-12-19,2023-12-20
35373,AID_053,C007,2024-01-26,59.886078,59.915305,54.915747,46.002797,119.745123,2.906624,33.0,0,2024-01-23,2024-01-25,1,1,0.0,1.0,11.0,2029.20,0.0,0,0,0,1,1,0,0,0,1,0,2024-01-27,2026-12-19,2023-12-20
35374,AID_053,C007,2024-01-29,68.465041,93.475450,74.019650,60.444282,39.562617,0.340188,19.0,0,2024-01-27,2024-01-27,1,1,0.0,0.0,20.0,477.72,0.0,0,1,0,0,0,0,0,1,0,1,2024-01-31,2026-12-19,2023-12-20
35375,AID_053,C007,2024-02-09,76.188597,67.602935,53.190737,72.409195,148.027449,0.202145,8.0,0,2024-02-09,2024-02-11,1,1,0.0,1.0,20.0,2367.20,0.0,0,0,0,1,1,0,0,0,0,1,2024-03-15,2026-12-19,2023-12-20
35376,AID_053,C007,2024-02-12,55.107102,42.520969,78.763349,72.600221,98.977276,3.202495,12.0,0,2024-02-09,2024-02-11,1,1,0.0,1.0,20.0,2367.20,0.0,0,0,0,1,1,0,0,0,0,1,2024-03-15,2026-12-19,2023-12-20
35377,AID_053,C007,2024-02-14,71.821831,58.419431,49.930381,51.123287,54.888314,1.633319,13.0,0,2024-02-09,2024-02-11,1,1,0.0,1.0,20.0,2367.20,0.0,0,0,0,1,1,0,0,0,0,1,2024-03-15,2026-12-19,2023-12-20


#### Save final_ads

In [22]:
#### Save final_ads
#
# Save the processed analytical dataset to disk for reproducibility and efficiency.
# This allows us to reload the data without redoing the expensive merging operations.
# Statistical importance: Preserves the exact data structure used for survival modeling.
os.makedirs('base_data', exist_ok=True)

with open('base_data/final_ads.pkl', 'wb') as f:
    pickle.dump(final_ads, f)
print("Final shape of the final_ads dataset is : ", final_ads.shape)

Final shape of the final_ads dataset is :  (40053, 33)


## Survaival Model Pipeline

This section implements the complete survival analysis workflow:
1. Feature Engineering: Create time-dependent covariates for survival modeling
2. Model Selection: Kaplan-Meier (baseline) → Cox PH (interpretability) → Random Survival Forest (final)
3. Feature Selection: Statistical methods to identify most predictive features
4. Model Evaluation: Concordance index and calibration assessment
5. Prediction: Generate failure probabilities for assets

Statistical approach: Progressive modeling from simple to complex, ensuring
each step improves predictive performance while maintaining interpretability.

In [23]:
# Create a working copy of the dataset and free up memory
# This allows us to modify the data without affecting the original merged dataset
df = final_ads.copy()
del final_ads

### Survival Analysis Feature Engineering

Now that we have our merged analytical dataset, we need to engineer features
specifically for survival analysis. This includes:
1. Time-dependent covariates (telemetry trends, lag features)
2. Static covariates (asset characteristics)
3. Event indicators (failure occurrences)
4. Time-to-event calculations

Statistical importance: Proper feature engineering is crucial for survival models
as they need to capture how asset conditions evolve over time and how these
changes affect the hazard of failure.

#### Feature Layer

Feature engineering is critical for survival analysis as it creates the time-dependent
covariates that capture how asset conditions evolve toward failure. We'll create:
1. Temporal features: Time since last ticket, ticket flags
2. Rolling statistics: Moving averages to capture recent trends
3. Trend features: Slopes to capture directional changes
4. Lag features: Historical values for autoregressive patterns
5. Event indicators: Failure occurrence signals

Statistical importance: These features enable the survival model to understand
the temporal dynamics leading to asset failure.

In [24]:
# Define the grouping levels for time-dependent feature creation
# We group by asset_id and client_id to ensure features are calculated
# independently for each asset-client combination
level = ['asset_id', 'client_id']

In [25]:
# Sort data by asset, client, and date to ensure proper temporal ordering
# Critical for survival analysis as features must be calculated in chronological order
df = df.sort_values(level + ['telemetry_record_date'])

# Convert date columns to datetime format for proper temporal calculations
# Essential for accurate time-based feature engineering
df['telemetry_record_date'] = pd.to_datetime(df['telemetry_record_date'])
df['ticket_create_date'] = pd.to_datetime(df['ticket_create_date'])
df['next_ticket_create_date'] = pd.to_datetime(df['next_ticket_create_date'])

In [26]:
# Create ticket-related temporal features
# These features capture the maintenance context which is crucial for survival analysis

# Days since last ticket: Indicates how long it's been since the last maintenance event
# Statistical importance: Longer periods without maintenance may increase failure risk
df['days_since_last_ticket'] = (
    df['telemetry_record_date'] - df['ticket_create_date']
).dt.days

# Ticket flag: Binary indicator of whether there's an active ticket
# Statistical importance: Active tickets indicate ongoing issues that may precede failure
df['ticket_flag'] = df['ticket_create_date'].notna().astype(int)

In [27]:
# Create rolling window features (core signals for survival analysis)
# Rolling statistics capture recent performance trends that are predictive of failure
# Multiple window sizes (3, 7, 14, 28 days) capture both short-term and long-term patterns

WINDOWS = [3, 7, 14, 28]

for w in WINDOWS:
    # Performance metrics rolling averages
    # Statistical importance: Recent average performance indicates current asset health
    df[f'cpu_mean_{w}'] = df.groupby(level)['tel_cpu_utilization_percent']\
        .transform(lambda x: x.rolling(w).mean())

    df[f'mem_mean_{w}'] = df.groupby(level)['tel_memory_utilization_percent']\
        .transform(lambda x: x.rolling(w).mean())

    df[f'disk_util_mean_{w}'] = df.groupby(level)['tel_disk_utilization_percent']\
        .transform(lambda x: x.rolling(w).mean())

    df[f'temp_mean_{w}'] = df.groupby(level)['tel_temperature_celsius']\
        .transform(lambda x: x.rolling(w).mean())

    df[f'net_lat_mean_{w}'] = df.groupby(level)['tel_network_latency_ms']\
        .transform(lambda x: x.rolling(w).mean())

    df[f'pack_loss_mean_{w}'] = df.groupby(level)['tel_packet_loss_percent']\
        .transform(lambda x: x.rolling(w).mean())

    # Maintenance-related rolling sums
    # Statistical importance: Accumulated maintenance burden indicates asset stress
    df[f'ticket_rolling_{w}'] = df.groupby(level)['ticket_flag']\
        .transform(lambda x: x.rolling(w).sum())

    df[f'error_sum_{w}'] = df.groupby(level)['tel_error_count']\
        .transform(lambda x: x.rolling(w).sum())

    df[f'ticket_reopen_sum_{w}'] = df.groupby(level)['t_ticket_reopened_count']\
        .transform(lambda x: x.rolling(w).sum())

    # Issue type rolling sums (specific problem indicators)
    # Statistical importance: Different issue types have different failure predictive power
    df[f'crash_sum_{w}'] = df.groupby(level)['t_Crash_issue_count']\
        .transform(lambda x: x.rolling(w).sum())

    df[f'slow_sum_{w}'] = df.groupby(level)['t_Slow_issue_count']\
        .transform(lambda x: x.rolling(w).sum())

    df[f'hard_cat_sum_{w}'] = df.groupby(level)['t_Hardware_category_count']\
        .transform(lambda x: x.rolling(w).sum())

    print(f'Window {w} Rolling features done')

Window 3 Rolling features done
Window 7 Rolling features done
Window 14 Rolling features done
Window 28 Rolling features done


In [28]:
# Create trend features using rolling slopes
# Trends capture the directional change in metrics, which are strong failure predictors
# Positive trends in performance metrics (CPU, memory, temperature) often precede failure

def rolling_slope(x):
    """Calculate the slope of a rolling window to capture trends"""
    return np.polyfit(range(len(x)), x, 1)[0] if len(x) > 1 else 0

WINDOWS = [3, 7, 14, 28]

for w in WINDOWS:
    # Performance metric trends
    # Statistical importance: Increasing trends indicate deteriorating asset condition
    df[f'cpu_trend_{w}'] = df.groupby(level)['tel_cpu_utilization_percent']\
        .transform(lambda x: x.rolling(w).apply(rolling_slope, raw=False))

    df[f'mem_trend_{w}'] = df.groupby(level)['tel_memory_utilization_percent']\
        .transform(lambda x: x.rolling(w).apply(rolling_slope, raw=False))

    df[f'disk_util_trend_{w}'] = df.groupby(level)['tel_disk_utilization_percent']\
        .transform(lambda x: x.rolling(w).apply(rolling_slope, raw=False))

    df[f'temp_trend_{w}'] = df.groupby(level)['tel_temperature_celsius']\
        .transform(lambda x: x.rolling(w).apply(rolling_slope, raw=False))

    df[f'net_lat_trend_{w}'] = df.groupby(level)['tel_network_latency_ms']\
        .transform(lambda x: x.rolling(w).apply(rolling_slope, raw=False))

    df[f'pack_loss_trend_{w}'] = df.groupby(level)['tel_packet_loss_percent']\
        .transform(lambda x: x.rolling(w).apply(rolling_slope, raw=False))

    print(f'Window {w} Trends done')

Window 3 Trends done
Window 7 Trends done
Window 14 Trends done
Window 28 Trends done


In [29]:
# Create lag features to capture historical values
# Lag features capture autoregressive patterns and delayed effects
# Different lag periods capture both immediate and historical influences

LAGS = [1, 3, 7, 14, 28]

for lag in LAGS:
    # Performance metric lags
    # Statistical importance: Previous performance levels influence current failure risk
    df[f'cpu_lag_{lag}'] = df.groupby(level)['tel_cpu_utilization_percent'].shift(lag)
    df[f'mem_lag_{lag}'] = df.groupby(level)['tel_memory_utilization_percent'].shift(lag)
    df[f'disk_util_lag_{lag}'] = df.groupby(level)['tel_disk_utilization_percent'].shift(lag)
    df[f'temp_lag_{lag}'] = df.groupby(level)['tel_temperature_celsius'].shift(lag)
    df[f'net_lat_lag_{lag}'] = df.groupby(level)['tel_network_latency_ms'].shift(lag)
    df[f'pack_loss_lag_{lag}'] = df.groupby(level)['tel_packet_loss_percent'].shift(lag)

    # Issue count lags
    # Statistical importance: Recent issues may have delayed effects on failure probability
    df[f'error_lag_{lag}'] = df.groupby(level)['tel_error_count'].shift(lag)
    df[f'ticket_reopen_lag_{lag}'] = df.groupby(level)['t_ticket_reopened_count'].shift(lag)
    df[f'crash_lag_{lag}'] = df.groupby(level)['t_Crash_issue_count'].shift(lag)
    df[f'slow_lag_{lag}'] = df.groupby(level)['t_Slow_issue_count'].shift(lag)
    df[f'hard_cat_lag_{lag}'] = df.groupby(level)['t_Hardware_category_count'].shift(lag)
    print(f'{lag} Lag features done')

1 Lag features done
3 Lag features done
7 Lag features done
14 Lag features done
28 Lag features done


In [30]:
# Create the failure event indicator (target variable for survival analysis)
# This is the key event indicator that survival models will predict
# Multiple failure sources create a comprehensive failure definition

df['failure_event'] = ((df['tel_failure_count'] > 0) | \
                       (df['t_Failure_issue_count'] > 0) | \
                       (pd.to_datetime(df['telemetry_record_date']) > pd.to_datetime(df['a_warranty_expiry_date']))
                      ).astype(int)

# Statistical reasoning:
# 1. tel_failure_count > 0: Direct telemetry-detected failures
# 2. t_Failure_issue_count > 0: User-reported failure tickets
# 3. Warranty expiry: Assets beyond warranty are considered "failed" from business perspective
# This comprehensive definition captures both technical and business failure perspectives

In [31]:
# Calculate duration (time-to-event) for survival analysis
# Duration represents the time from installation to the current observation date
# This is the time component needed for survival modeling

df['duration'] = (
    pd.to_datetime(df['telemetry_record_date']) - pd.to_datetime(df['a_installation_date'])
).dt.days

# Remove data leakage by ensuring duration is non-negative
# Statistical importance: Negative durations would indicate data quality issues
# and would corrupt the survival analysis
df = df[df['duration'] >= 0]

In [32]:
# Prepare the final modeling dataset
# Remove leakage variables and keep only predictive features for survival analysis

# Get all numeric columns (potential features for survival modeling)
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()

# Remove columns that would cause data leakage
# These columns contain information about the target variable (failure)
remove_cols = {'t_Failure_issue_count', 'tel_failure_count'}

# Keep only features that don't leak target information
keep_cols = [
    col for col in numeric_cols if col not in remove_cols
]

model_df = df[keep_cols].copy()

# Clean up memory and save the prepared modeling dataset
del df # clean memory

os.makedirs('base_data', exist_ok=True)

with open('base_data/model_data.pkl', 'wb') as f:
    pickle.dump(model_df, f)
print("Final shape of the model_df dataset is : ", model_df.shape)

Final shape of the model_df dataset is :  (40053, 154)


### Model Selection Strategy

Progressive modeling approach for optimal survival analysis:
1. Kaplan-Meier (KM): Non-parametric baseline for understanding survival patterns
2. Cox Proportional Hazards: Semi-parametric model for interpretability and covariate effects
3. Random Survival Forest (RSF): Machine learning approach for maximum predictive accuracy

Statistical reasoning: Start simple, add complexity gradually, ensuring each step
improves understanding and predictive performance while maintaining interpretability.

In [33]:
def survival_feature_selection_model_pipeline(
    df,
    duration_col='duration',
    event_col='failure_event',
    corr_threshold=0.75,
    pval_threshold=0.05,
    top_n=20
):
    """
    Comprehensive feature selection pipeline for survival analysis

    Statistical approach: Multi-stage filtering to identify most predictive features
    while avoiding overfitting and maintaining model interpretability

    Stages:
    1. Variance filter: Remove features with no variation
    2. Correlation filter: Remove highly correlated features to avoid multicollinearity
    3. Kaplan-Meier filter: Keep features with significant survival differences
    4. Cox regression filter: Keep features with significant hazard ratios
    5. Random Forest importance: Select top features by predictive power
    """
    # ---------------------------
    # STEP 1: Keep numeric only
    # ---------------------------
    df = df.select_dtypes(include=['number']).copy()

    # Ensure event column is properly formatted (0/1)
    df[event_col] = (
        pd.to_numeric(df[event_col], errors='coerce')
        .fillna(0)
        .astype(int)
        .clip(0, 1)
    )

    # Remove invalid observations
    df = df[df[duration_col] > 0].dropna()

    FEATURES = [col for col in df.columns if col not in [duration_col, event_col]]
    print(f"Initial features: {len(FEATURES)}")

    # ---------------------------
    # STEP 2: Low variance filter
    # ---------------------------
    # Remove features with no variation (cannot predict anything)
    low_var = [col for col in FEATURES if df[col].nunique() <= 1]
    df = df.drop(columns=low_var)
    FEATURES = [col for col in FEATURES if col not in low_var]
    print(f"After variance filter: {len(FEATURES)}\n Variance filter logic: Remove features with no variation\n")

    # ---------------------------
    # STEP 3: Correlation filter
    # ---------------------------
    # Remove highly correlated features to avoid multicollinearity
    # Statistical importance: Correlated features can destabilize coefficient estimates
    corr_matrix = df[FEATURES].corr().abs()

    to_drop = set()
    for i in range(len(FEATURES)):
        for j in range(i):
            if corr_matrix.iloc[i, j] > corr_threshold:
                to_drop.add(corr_matrix.columns[i])

    df = df.drop(columns=list(to_drop))
    FEATURES = [col for col in FEATURES if col not in to_drop]
    print(f"After correlation filter: {len(FEATURES)}\n Correlation filter logic: Remove highly correlated features to avoid multicollinearity\n")

    # ---------------------------
    # STEP 4: Kaplan-Meier filter
    # ---------------------------
    # Use log-rank test to identify features with significant survival differences
    # Statistical importance: Features that don't separate survival curves have limited predictive value
    km_selected = []
    pvals = {}

    for col in FEATURES:
        median_val = df[col].median()
        high_mask = df[col] > median_val

        # Ensure sufficient sample size in both groups
        if high_mask.sum() < 10 or (~high_mask).sum() < 10:
            continue

        # Log-rank test for survival differences
        results = logrank_test(
            df[high_mask][duration_col],
            df[~high_mask][duration_col],
            event_observed_A=df[high_mask][event_col],
            event_observed_B=df[~high_mask][event_col]
        )

        pvals[col] = results.p_value

        if results.p_value < pval_threshold:
            km_selected.append(col)

    print(f"After KM filter: {len(km_selected)}\n Kaplan-Meier filter logic: Keep features with significant survival differences\n")

    # Handle case where no features pass the filter
    if len(km_selected) == 0:
        print("No features passed KM filter — relaxing condition")
        km_selected = FEATURES

    # ---------------------------
    # STEP 5: Cox regression filter
    # ---------------------------
    # Use Cox model to identify features with significant hazard ratios
    # Statistical importance: Cox model accounts for censoring and provides hazard ratios
    cox_df = df[[duration_col, event_col] + km_selected].copy()

    cox = CoxPHFitter(penalizer=0.1, l1_ratio=0.5)
    cox.fit(cox_df, duration_col=duration_col, event_col=event_col)

    cox_summary = cox.summary.sort_values("p")

    cox_selected = cox_summary[cox_summary["p"] < pval_threshold].index.tolist()

    if len(cox_selected) == 0:
        print("No features passed Cox filter — using KM selected")
        cox_selected = km_selected

    print(f"After Cox filter: {len(cox_selected)}\n Cox regression filter: Keep features with significant hazard ratios\n")

    # ---------------------------
    # STEP 6: Random Forest permutation importance
    # ---------------------------
    # Use permutation importance to select top predictive features
    # Statistical importance: Captures non-linear relationships and interactions
    X = df[cox_selected]

    y = Surv.from_dataframe(
        event=event_col,
        time=duration_col,
        data=df.assign(**{event_col: df[event_col].astype(bool)})
    )

    rsf = RandomSurvivalForest(
        n_estimators=50,
        max_depth=10,
        min_samples_split=20,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    )

    rsf.fit(X, y)

    perm = permutation_importance(
        rsf,
        X,
        y,
        n_repeats=5,
        random_state=42,
        n_jobs=-1
    )

    importances = pd.Series(perm.importances_mean, index=cox_selected)
    top_features = importances.sort_values(ascending=False).head(top_n)

    print(f"Final selected features: {len(top_features)}\n Random Forest importance: Select top features by predictive power")

    return {
        "selected_features": top_features.index.tolist(),
        "feature_importance": top_features,
        "km_pvalues": pvals,
        "cox_summary": cox_summary
    }

In [34]:
# Execute the comprehensive feature selection pipeline
# This identifies the most predictive features for survival analysis
# Statistical importance: Feature selection prevents overfitting and improves model interpretability
model_result = survival_feature_selection_model_pipeline(model_df)

Initial features: 152
After variance filter: 147
 Variance filter logic: Remove features with no variation

After correlation filter: 118
 Correlation filter logic: Remove highly correlated features to avoid multicollinearity

After KM filter: 24
 Kaplan-Meier filter logic: Keep features with significant survival differences

After Cox filter: 5
 Cox regression filter: Keep features with significant hazard ratios

Final selected features: 5
 Random Forest importance: Select top features by predictive power


In [35]:
# Display the final selected features
# These are the most predictive features identified through the multi-stage selection process
# Statistical importance: These features have proven predictive power across multiple methods
model_result['selected_features']

['t_Crash_issue_count',
 't_Slow_issue_count',
 't_Access_issue_count',
 'tel_cpu_utilization_percent',
 't_ticket_count']

In [36]:
# Display feature importance scores from Random Survival Forest
# Higher importance indicates stronger predictive power for asset failure
# Statistical importance: These scores guide feature prioritization and model interpretation
model_result['feature_importance']

,0
t_Crash_issue_count,0.153751
t_Slow_issue_count,0.144648
t_Access_issue_count,0.141761
tel_cpu_utilization_percent,0.115036
t_ticket_count,0.007551


In [37]:
# Display Kaplan-Meier log-rank test p-values
# These p-values indicate which features significantly separate survival curves
# Statistical importance: Lower p-values suggest stronger association with survival outcomes
model_result['km_pvalues']

{'tel_cpu_utilization_percent': np.float64(6.304631820085475e-216),
 'tel_memory_utilization_percent': np.float64(0.8545601971024879),
 'tel_disk_utilization_percent': np.float64(0.13131683949669104),
 'tel_temperature_celsius': np.float64(0.42490545766907006),
 'tel_network_latency_ms': np.float64(0.9858319324417172),
 'tel_packet_loss_percent': np.float64(0.25866139392651444),
 'tel_error_count': np.float64(7.966359813821273e-06),
 't_ticket_count': np.float64(6.928676237968737e-25),
 't_escalations_count': np.float64(0.0030504375149425964),
 't_ticket_reopened_count': np.float64(2.6684795641244946e-09),
 't_avg_first_response_time_minutes': np.float64(0.34215129569567165),
 't_avg_resolution_time_minutes': np.float64(0.5569252082025935),
 't_P1_ticket_count': np.float64(0.2748315421593732),
 't_P2_ticket_count': np.float64(0.01205176639880687),
 't_P3_ticket_count': np.float64(0.6021646755610097),
 't_Access_issue_count': np.float64(8.28700517262751e-256),
 't_Crash_issue_count': np

In [38]:
# Display Cox regression summary
# Shows hazard ratios, confidence intervals, and p-values for each feature
# Statistical importance: Hazard ratios > 1 indicate increased failure risk, < 1 indicate protective effect
model_result['cox_summary']

,coef,exp(coef),se(coef),coef lower 95%,coef upper 95%,exp(coef) lower 95%,exp(coef) upper 95%,cmp to,z,p,-log2(p)
covariate,,,,,,,,,,,
t_Crash_issue_count,-8.596322e-01,0.423318,2.373507e-02,-9.061521e-01,-8.131123e-01,0.404076,0.443476,0.0,-36.217815,3.192342e-287,951.718748
t_Access_issue_count,-8.350714e-01,0.433844,2.393736e-02,-8.819877e-01,-7.881550e-01,0.413959,0.454683,0.0,-34.885689,1.225429e-266,883.339586
t_Slow_issue_count,-8.281142e-01,0.436872,2.401763e-02,-8.751879e-01,-7.810405e-01,0.416784,0.457929,0.0,-34.479435,1.631334e-260,862.995252
tel_cpu_utilization_percent,1.970478e-02,1.019900,6.524591e-04,1.842598e-02,2.098357e-02,1.018597,1.021205,0.0,30.200780,2.313356e-200,663.175631
t_ticket_count,1.576856e-01,1.170798,7.370177e-02,1.323276e-02,3.021384e-01,1.013321,1.352748,0.0,2.139509,3.239449e-02,4.948108
cpu_trend_7,3.772760e-08,1.000000,5.384272e-06,-1.051525e-05,1.059071e-05,0.999989,1.000011,0.0,0.007007,9.944093e-01,0.008088
cpu_mean_3,1.226176e-08,1.000000,1.750074e-06,-3.417821e-06,3.442345e-06,0.999997,1.000003,0.0,0.007006,9.944097e-01,0.008088
cpu_trend_14,6.655217e-08,1.000000,1.006788e-05,-1.966613e-05,1.979924e-05,0.999980,1.000020,0.0,0.006610,9.947257e-01,0.007629
tel_error_count,7.454987e-09,1.000000,1.171103e-06,-2.287865e-06,2.302775e-06,0.999998,1.000002,0.0,0.006366,9.949209e-01,0.007346


#### Model Fitting with Selected Features

Now we fit the final Random Survival Forest model using only the most predictive features
This approach maximizes predictive accuracy while avoiding overfitting

Statistical advantages:
- Reduced dimensionality prevents curse of dimensionality
- Feature selection removes noise and irrelevant variables
- Focus on most predictive features improves generalization

In [39]:
# Prepare data for final Random Survival Forest model
# Use only the selected features to maximize predictive performance

duration_col='duration'
event_col='failure_event'

# Extract selected features from the modeling dataset
X = model_df[model_result['selected_features']]

# Create survival data structure required by scikit-survival
# This format handles censored data properly for survival modeling
y = Surv.from_dataframe(
        event=event_col,
        time=duration_col,
        data=model_df.assign(**{event_col: model_df[event_col].astype(bool)})
    )

# Initialize Random Survival Forest with optimized parameters
# Statistical considerations:
# - n_estimators=50: Balance between accuracy and computational efficiency
# - max_depth=10: Prevent overfitting while capturing complex relationships
# - min_samples_split=20: Ensure sufficient data for reliable splits
# - min_samples_leaf=10: Prevent overfitting on small subsets
rsf = RandomSurvivalForest(
        n_estimators=50,
        max_depth=10,
        min_samples_split=20,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1  # Use all available cores for faster training
    )

# Fit the model to learn survival patterns
rsf.fit(X, y)

,n_estimators,50
,max_depth,10
,min_samples_split,20
,min_samples_leaf,10
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,bootstrap,True
,oob_score,False
,n_jobs,-1
,random_state,42


In [40]:
# Generate risk scores for all observations
# These predictions represent the relative risk ranking for each asset at each time point
# Statistical importance: Higher risk scores indicate higher probability of imminent failure

pred = rsf.predict(model_df[model_result['selected_features']])

### Model Evaluation

Evaluate the survival model performance using appropriate metrics:
1. Concordance Index (C-index): Measures discriminative ability
2. Calibration: Assesses predicted vs. actual survival probabilities

Statistical importance: Proper evaluation ensures the model can reliably
predict asset failure times and provides confidence in business decisions.

In [41]:
# Calculate Concordance Index (C-index) - the primary evaluation metric for survival models
# C-index measures the model's ability to correctly rank survival times
# Statistical interpretation:
# - 1.0 = Perfect prediction (all higher-risk subjects fail earlier)
# - 0.5 = Random prediction (no better than chance)
# - < 0.5 = Worse than random (model is inverted)

c_index = concordance_index_censored(
    model_df[event_col].astype(bool),  # Event indicator (censoring information)
    model_df[duration_col].astype(bool),  # Observed times
    pred  # Predicted risk scores
)

print("C-index:", c_index[0])

C-index: 0.9889691816638019


In [42]:
# Detailed interpretation of C-index results
print(f"\n=== C-index Results Interpretation ===")
print(f"c_index[0]: Actual Concordance Index value = {c_index[0]:.6f}")
print(f"  - Statistical meaning: Probability that for any two comparable subjects,")
print(f"    the one with higher predicted risk score experiences the event earlier")
print(f"  - Business interpretation: Higher values indicate better asset failure prediction")
print(f"  - Our value ({c_index[0]:.3f}) indicates {'excellent' if c_index[0] > 0.9 else 'good' if c_index[0] > 0.8 else 'moderate' if c_index[0] > 0.7 else 'poor'} discriminative ability")

print(f"\nc_index[1]: Number of concordant pairs = {c_index[1]:,}")
print(f"  - Statistical meaning: Pairs where predicted risk ordering matches actual survival ordering")
print(f"  - Higher numbers relative to discordant pairs indicate better performance")

print(f"\nc_index[2]: Number of discordant pairs = {c_index[2]:,}")
print(f"  - Statistical meaning: Pairs where predicted risk ordering contradicts actual survival")
print(f"  - Lower numbers are desirable for good model performance")

print(f"\nc_index[3]: Number of tied event times = {c_index[3]:,}")
print(f"  - Statistical meaning: Subjects with identical observed event times")
print(f"  - Important for understanding data structure and potential challenges")

print(f"\nc_index[4]: Total number of comparable pairs = {c_index[4]:,}")
print(f"  - Statistical meaning: All pairs that can be evaluated considering censoring")
print(f"  - Provides context for the concordance calculation")

print(f"\n=== Performance Summary ===")
print(f"Concordance Rate: {(c_index[1] / c_index[4]) * 100:.2f}%")
print(f"Discordance Rate: {(c_index[2] / c_index[4]) * 100:.2f}%")
print(f"Tie Rate: {(c_index[3] / c_index[4]) * 100:.2f}%")


=== C-index Results Interpretation ===
c_index[0]: Actual Concordance Index value = 0.988969
  - Statistical meaning: Probability that for any two comparable subjects,
    the one with higher predicted risk score experiences the event earlier
  - Business interpretation: Higher values indicate better asset failure prediction
  - Our value (0.989) indicates excellent discriminative ability

c_index[1]: Number of concordant pairs = 367,477,117
  - Statistical meaning: Pairs where predicted risk ordering matches actual survival ordering
  - Higher numbers relative to discordant pairs indicate better performance

c_index[2]: Number of discordant pairs = 4,094,379
  - Statistical meaning: Pairs where predicted risk ordering contradicts actual survival
  - Lower numbers are desirable for good model performance

c_index[3]: Number of tied event times = 8,914
  - Statistical meaning: Subjects with identical observed event times
  - Important for understanding data structure and potential chal

In [43]:
# Generate survival functions for visualization and interpretation
# Survival functions show the probability of survival over time for each asset
# Statistical importance: These functions provide complete survival curves,
# not just point predictions, enabling comprehensive risk assessment

surv_fn = rsf.predict_survival_function(X)
surv_fn

array([StepFunction(x=array([0.000e+00, 1.000e+00, 2.000e+00, ..., 1.092e+03, 1.093e+03,
              1.094e+03]), y=array([1.       , 1.       , 1.       , ..., 0.7918527, 0.7918527,
              0.7918527]), a=1.0, b=0.0)                                                       ,
       StepFunction(x=array([0.000e+00, 1.000e+00, 2.000e+00, ..., 1.092e+03, 1.093e+03,
              1.094e+03]), y=array([1.        , 1.        , 1.        , ..., 0.99609817, 0.99609817,
              0.99609817]), a=1.0, b=0.0)                                                           ,
       StepFunction(x=array([0.000e+00, 1.000e+00, 2.000e+00, ..., 1.092e+03, 1.093e+03,
              1.094e+03]), y=array([1.        , 1.        , 1.        , ..., 0.64066883, 0.64066883,
              0.64066883]), a=1.0, b=0.0)                                                           ,
       ...,
       StepFunction(x=array([0.000e+00, 1.000e+00, 2.000e+00, ..., 1.092e+03, 1.093e+03,
              1.094e+03]), y=arra

### Merging Prediction to the Asset Client Level Latest Telemetry Information

In [44]:
# --- Step 1 & 2: Load and align identifiers with model_df's current state ---
# Reload original_ads to extract identifiers
original_ads_path = 'base_data/final_ads.pkl'
with open(original_ads_path, 'rb') as f:
    full_ads = pickle.load(f)

# Replicate the processing steps that created the 'df' from which 'model_df' was derived
# This ensures correct index alignment after filtering
temp_df_for_ids = full_ads.copy()

# Convert date columns to datetime format (from cell 15b008aa)
temp_df_for_ids['telemetry_record_date'] = pd.to_datetime(temp_df_for_ids['telemetry_record_date'])
temp_df_for_ids['ticket_create_date'] = pd.to_datetime(temp_df_for_ids['ticket_create_date'])
temp_df_for_ids['next_ticket_create_date'] = pd.to_datetime(temp_df_for_ids['next_ticket_create_date'])

# Recreate 'days_since_last_ticket' and 'ticket_flag' (from cell 9c15023f)
temp_df_for_ids['days_since_last_ticket'] = (
    temp_df_for_ids['telemetry_record_date'] - temp_df_for_ids['ticket_create_date']
).dt.days
temp_df_for_ids['ticket_flag'] = temp_df_for_ids['ticket_create_date'].notna().astype(int)

# Create the failure event indicator (from cell acbb108a)
temp_df_for_ids['failure_event'] = ((temp_df_for_ids['tel_failure_count'] > 0) |
                                     (temp_df_for_ids['t_Failure_issue_count'] > 0) |
                                     (pd.to_datetime(temp_df_for_ids['telemetry_record_date']) > pd.to_datetime(temp_df_for_ids['a_warranty_expiry_date']))
                                    ).astype(int)

# Calculate duration (time-to-event) (from cell 7dc779c2)
temp_df_for_ids['duration'] = (
    pd.to_datetime(temp_df_for_ids['telemetry_record_date']) - pd.to_datetime(temp_df_for_ids['a_installation_date'])
).dt.days

# Remove data leakage by ensuring duration is non-negative (from cell 7dc779c2)
temp_df_for_ids = temp_df_for_ids[temp_df_for_ids['duration'] >= 0]

# Extract identifiers and relevant columns, aligning with model_df's current index
identifiers_df = temp_df_for_ids.loc[model_df.index, ['asset_id', 'client_id', 'telemetry_record_date', 'duration', 'a_installation_date']].copy()

# Add the predicted risk scores to this DataFrame
identifiers_df['predicted_risk_score'] = pred

del temp_df_for_ids


In [45]:
# --- Step 3: Filter to get the latest prediction for each asset-client pair ---
# Sort to ensure the latest telemetry_record_date comes first for each asset-client
latest_predictions_df = identifiers_df.sort_values(
    by=['asset_id', 'client_id', 'telemetry_record_date'],
    ascending=[True, True, False]  # Latest date first
).drop_duplicates(subset=['asset_id', 'client_id'], keep='first').reset_index(drop=True)

# Get the original integer positions (iloc) from model_df for these latest records
# This is crucial for correctly mapping to the 'surv_fn' array
index_to_iloc = {idx: i for i, idx in enumerate(model_df.index)}
iloc_for_latest = [index_to_iloc[idx] for idx in identifiers_df.sort_values(
    by=['asset_id', 'client_id', 'telemetry_record_date'],
    ascending=[True, True, False]
).drop_duplicates(subset=['asset_id', 'client_id'], keep='first').index.values]

# Now, get the survival functions for these specific rows
surv_fns_for_latest = [surv_fn[i] for i in iloc_for_latest]

In [46]:
# --- Step 4: Calculate Predicted Remaining Time to Failure ---
def get_remaining_time_to_event(survival_function, current_duration_days, threshold=0.5):
    """
    Estimates the remaining time until the event (failure) based on a survival function.
    Args:
        survival_function (sksurv.functions.StepFunction): The survival function.
        current_duration_days (int): The current duration of the asset in days from installation.
        threshold (float): The survival probability threshold to consider as 'event time'.
                           e.g., 0.5 for median survival time.
    Returns:
        tuple: (Predicted remaining days until event, Probability of failure at that time)
               or (np.nan, np.nan) if event doesn't occur before max time.
    """
    times = survival_function.x
    probabilities = survival_function.y

    # Find the first time point where survival probability drops below the threshold
    event_idx = np.where(probabilities < threshold)[0]

    if len(event_idx) > 0:
        predicted_total_duration = times[event_idx[0]]
        remaining_days = predicted_total_duration - current_duration_days
        # Probability of failure is 1 - survival probability at the predicted event time
        failure_probability = 1.0 - probabilities[event_idx[0]]
        return max(0, int(remaining_days)), failure_probability
    else:
        # If survival probability never drops below threshold, asset might not fail within observed max time
        return np.nan, np.nan

# Add 'predicted_remaining_days_to_failure' and 'predicted_risk_probability' to latest_predictions_df
results = [
    get_remaining_time_to_event(sf, row['duration']) for sf, (_, row) in zip(surv_fns_for_latest, latest_predictions_df.iterrows())
]
latest_predictions_df['predicted_remaining_days_to_failure'] = [res[0] for res in results]
latest_predictions_df['predicted_risk_probability'] = [res[1] for res in results]

In [47]:
# --- Step 5: Filter for assets needing replacement based on lead time ---

latest_predictions_df = latest_predictions_df.merge(latest_inventory[['asset_id', 'i_lead_time_days']], how='left', on='asset_id')

assets_to_order_now = latest_predictions_df[
    (latest_predictions_df['predicted_remaining_days_to_failure'].notna()) &
    (latest_predictions_df['predicted_remaining_days_to_failure'] <= latest_predictions_df['i_lead_time_days'])
].copy()

# Add a column for the date by which the replacement asset should be installed
assets_to_order_now['replacement_needed_by_date'] = assets_to_order_now['telemetry_record_date'] + \
                                                    pd.to_timedelta(assets_to_order_now['predicted_remaining_days_to_failure'], unit='D')

# Display the final results
final_output_columns = [
    'asset_id',
    'client_id',
    'telemetry_record_date',
    'i_lead_time_days',
    'predicted_risk_probability',
    'predicted_remaining_days_to_failure',
    'replacement_needed_by_date'
]

print(f"Assets predicted to fail within the lead time (sorted by imminent failure):")
display(assets_to_order_now[final_output_columns].sort_values('predicted_remaining_days_to_failure').head(10))

print(f"\nTotal assets predicted to fail within the next lead time days: {assets_to_order_now.shape[0]}")

Assets predicted to fail within the lead time (sorted by imminent failure):


,asset_id,client_id,telemetry_record_date,i_lead_time_days,predicted_risk_probability,predicted_remaining_days_to_failure,replacement_needed_by_date
6,AID_002,C012,2023-12-31,10,0.501025,0.0,2023-12-31
8,AID_002,C015,2024-12-30,10,0.500650,0.0,2024-12-30
14,AID_003,C024,2023-12-28,6,0.507250,0.0,2023-12-28
23,AID_005,C016,2025-12-31,9,0.500689,0.0,2025-12-31
28,AID_006,C008,2023-12-26,9,0.502909,0.0,2023-12-26
30,AID_006,C015,2024-12-26,9,0.501316,0.0,2024-12-26
41,AID_008,C015,2025-12-31,13,0.502423,0.0,2025-12-31
42,AID_008,C017,2024-12-28,13,0.502825,0.0,2024-12-28
44,AID_008,C019,2023-12-31,13,0.530784,0.0,2023-12-31
50,AID_009,C013,2024-12-26,5,0.501662,0.0,2024-12-26



Total assets predicted to fail within the next lead time days: 128


### Save final_ads

In [48]:
# Save the processed predictions to disk for use in next step of Inventory Procurment ordeing Algorithm.
os.makedirs('base_data', exist_ok=True)

with open('base_data/assets_to_order_now.pkl', 'wb') as f:
    pickle.dump(assets_to_order_now, f)
print("Final shape of the final_ads dataset is : ", assets_to_order_now.shape)

Final shape of the final_ads dataset is :  (128, 10)


In [49]:
assets_to_order_now.columns

Index(['asset_id', 'client_id', 'telemetry_record_date', 'duration', 'a_installation_date', 'predicted_risk_score', 'predicted_remaining_days_to_failure', 'predicted_risk_probability', 'i_lead_time_days', 'replacement_needed_by_date'], dtype='object')

In [50]:
# =========================
# EXPORT: Asset Risk Scores + Telemetry to JSON
# =========================

import json
from pathlib import Path

JSON_DIR = Path("model_outputs")
JSON_DIR.mkdir(exist_ok=True)

# ── 1. Asset risk scores ──
# latest_predictions_df already exists from Step 3-5 of this notebook.
# It has: asset_id, client_id, predicted_risk_probability,
#          predicted_remaining_days_to_failure, telemetry_record_date

# Add display-friendly fields
export_df = latest_predictions_df.copy()

# Get asset metadata (device_type, model_number) if available
if 'assets' in dir():
    meta_cols = [c for c in ['asset_id','device_type','model_number','manufacturer',
                             'warranty_expiry_date'] if c in assets.columns]
    if meta_cols:
        meta = assets[meta_cols].drop_duplicates('asset_id')
        export_df = export_df.merge(meta, on='asset_id', how='left')

# Get lead time from inventory
if 'latest_inventory' in dir() and 'i_lead_time_days' in latest_inventory.columns:
    if 'i_lead_time_days' not in export_df.columns:
        export_df = export_df.merge(
            latest_inventory[['asset_id','i_lead_time_days']].drop_duplicates('asset_id'),
            on='asset_id', how='left'
        )

# Compute replacement_needed_by_date if not already present
if 'replacement_needed_by_date' not in export_df.columns:
    export_df['replacement_needed_by_date'] = (
        pd.to_datetime(export_df['telemetry_record_date']) +
        pd.to_timedelta(export_df['predicted_remaining_days_to_failure'].fillna(9999), unit='D')
    )

# Select columns for JSON
keep = ['asset_id', 'client_id', 'device_type', 'model_number',
        'predicted_risk_probability', 'predicted_remaining_days_to_failure',
        'replacement_needed_by_date', 'i_lead_time_days', 'telemetry_record_date']
keep = [c for c in keep if c in export_df.columns]

asset_risk = export_df[keep].to_dict(orient='records')

with open(JSON_DIR / "asset_risk_scores.json", "w") as f:
    json.dump(asset_risk, f, indent=2, default=str)

print(f"✅  asset_risk_scores.json: {len(asset_risk)} records → {JSON_DIR}")

# ── 2. Telemetry aggregation (for dashboard charts) ──
# Uses the telemetry DataFrame (typically called 'telemetry' or part of final_ads)

telemetry_source = None
if 'telemetry' in dir():
    telemetry_source = telemetry
elif 'final_ads' in dir():
    telemetry_source = final_ads

if telemetry_source is not None:
    # Find the telemetry columns
    cpu_col = [c for c in telemetry_source.columns if 'cpu' in c.lower() and 'util' in c.lower()]
    mem_col = [c for c in telemetry_source.columns if 'mem' in c.lower() and 'util' in c.lower()]
    temp_col = [c for c in telemetry_source.columns if 'temp' in c.lower()]
    ts_col = [c for c in telemetry_source.columns if 'record_date' in c.lower() or 'timestamp' in c.lower()]

    if cpu_col and ts_col:
        # Use the latest day's data, grouped by hour
        ts = pd.to_datetime(telemetry_source[ts_col[0]])
        latest_day = ts.max().date()
        mask = ts.dt.date == latest_day
        day_data = telemetry_source[mask].copy()
        day_data['_hour'] = ts[mask].dt.strftime("%H:00")

        def hourly_avg(col_list):
            if not col_list:
                return []
            col = col_list[0]
            return (day_data.groupby('_hour')[col].mean()
                    .round(0).reset_index()
                    .rename(columns={'_hour':'time', col:'value'})
                    .to_dict(orient='records'))

        telemetry_agg = {
            "cpu_load_trend": hourly_avg(cpu_col),
            "ram_trend": hourly_avg(mem_col),
            "thermal_trend": hourly_avg(temp_col),
            "fleet_kpis": {
                "avg_cpu_pct": round(float(telemetry_source[cpu_col[0]].mean())) if cpu_col else 0,
                "avg_memory_pct": round(float(telemetry_source[mem_col[0]].mean())) if mem_col else 0,
                "avg_temp_pct": round(float(telemetry_source[temp_col[0]].mean())) if temp_col else 0,
                "critical_alerts": int((export_df['predicted_risk_probability'] > 0.7).sum()),
            }
        }

        with open(JSON_DIR / "telemetry_agg.json", "w") as f:
            json.dump(telemetry_agg, f, indent=2, default=str)
        print(f"✅  telemetry_agg.json → {JSON_DIR}")
    else:
        print("⚠️  Could not find CPU/timestamp columns for telemetry aggregation")
else:
    print("⚠️  No telemetry DataFrame found. Skipping telemetry_agg.json")

print("Done.")

✅  asset_risk_scores.json: 385 records → model_outputs
✅  telemetry_agg.json → model_outputs
Done.
